# 04 — Modelo 1: Detección Binaria de Anomalías

Entrenamos un Random Forest para clasificar cada fila como `normal` o `anomalia`.

**Diferencia clave respecto a v3:**
- **v2:** split aleatorio 70/30 (`train_test_split`)
- **v3:** TimeSeriesSplit k=4 (sin data leakage temporal)

El split aleatorio puede introducir data leakage leve porque mezcla fechas del pasado y futuro en el mismo fold, pero es más simple de entender.

**Entrada:** `data/interim/03_datos_features.parquet`  
**Salida:** modelos en `data/models/`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y preparar X, y

In [ ]:
df_trabajo = pd.read_parquet(PARQUET_03)
print(f"Dataset cargado: {df_trabajo.shape}")

# Separar features (X) y etiqueta binaria (y)
excluir = COLUMNAS_EXCLUIR_FEATURES
columnas_features = [c for c in df_trabajo.columns if c not in excluir]
X = df_trabajo[columnas_features].select_dtypes(include='number').copy()
y = df_trabajo['etiqueta_deteccion'].copy()

print(f"Features: {X.shape[1]} columnas")
print(f"Distribución de clases:")
print(y.value_counts())


## 1. División train/test — split aleatorio 70/30
**v2** usa `train_test_split` con `random_state=42` para reproducibilidad.
Esto divide aleatoriamente las filas sin respetar el orden temporal (a diferencia de v3 que usa TimeSeriesSplit).

In [ ]:
if 'X' in locals() and 'y' in locals():
    # Dividir los datos en conjuntos de entrenamiento y prueba
    # test_size=0.3 significa que el 30% de los datos será para prueba, 70% para entrenamiento.
    # random_state es para asegurar la reproducibilidad de la división.
    # stratify=y es importante para clasificación, especialmente si hay desbalance de clases,
    # asegura que la proporción de clases en 'y' se mantenga en los conjuntos de ent. y prueba.
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=0.3, 
            random_state=42, 
            stratify=y # Recomendado para mantener proporción de clases
        )

        print("\nDatos divididos en conjuntos de entrenamiento y prueba:")
        print(f"Forma de X_train: {X_train.shape}, Forma de y_train: {y_train.shape}")
        print(f"Forma de X_test: {X_test.shape}, Forma de y_test: {y_test.shape}")

        print("\nDistribución de la variable objetivo en el conjunto de entrenamiento:")
        print(y_train.value_counts(normalize=True))
        print("\nDistribución de la variable objetivo en el conjunto de prueba:")
        print(y_test.value_counts(normalize=True))
        
    except Exception as e:
        print(f"Error durante la división de datos: {e}")
        print("Asegúrate de que X e y tengan el mismo número de muestras y no haya problemas con 'stratify'.")
        if len(y.value_counts()) < 2 :
                print("  Posible causa: La variable objetivo 'y' tiene menos de 2 clases, 'stratify' podría fallar.")
                print("  Intentando dividir sin stratify (menos ideal si hay desbalance):")
                try:
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
                    print("\n  División sin stratify completada.")
                    print(f"  Forma de X_train: {X_train.shape}, Forma de y_train: {y_train.shape}")
                    print(f"  Forma de X_test: {X_test.shape}, Forma de y_test: {y_test.shape}")
                except Exception as e2:
                    print(f"  Error en división sin stratify: {e2}")


else:
    print("Error: 'X' o 'y' no están definidos. Ejecuta el paso anterior.")

## 2. Imputar NaNs con SimpleImputer + indicadores
Rellena NaNs con la mediana y añade columnas binarias `missingindicator_X`. El hecho de tener NaN es en sí una señal de posible anomalía.

In [ ]:
if 'X_train' in locals() and 'X_test' in locals():
    print("Manejo de NaNs antes de la imputación:")
    print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
    print(f"NaNs en X_test: {X_test.isnull().sum().sum()}")

    # Crear el imputador usando la mediana Y AÑADIENDO COLUMNAS INDICADORAS
    imputer = SimpleImputer(strategy='median', add_indicator=True)

    # Ajustar el imputador SOLO con los datos de entrenamiento
    # Nota: fit no devuelve nada, solo configura el imputer
    imputer.fit(X_train) 
    
    # Aplicar la imputación (transformar) a los datos de entrenamiento y prueba
    # El resultado será un array de NumPy
    X_train_imputed_np = imputer.transform(X_train)
    X_test_imputed_np = imputer.transform(X_test)

    # Obtener los nombres de las nuevas características (originales + indicadoras)
    try:
        # Intenta obtener los nombres de las características, incluyendo las indicadoras
        feature_names_imputed = imputer.get_feature_names_out(input_features=X_train.columns)
    except AttributeError:
        print("Advertencia: imputer.get_feature_names_out no disponible. Usando nombres genéricos para indicadoras si es necesario.") 
        # Si no se puede obtener, usamos un enfoque alternativo
        feature_names_imputed = list(X_train.columns)
        # Ver cuántas columnas indicadoras se añadieron
        num_original_features = X_train.shape[1]
        num_indicator_features = X_train_imputed_np.shape[1] - num_original_features
        if num_indicator_features > 0:
            # Obtener índices de características que tenían NaNs y para las cuales se creó un indicador
            indicator_indices = imputer.indicator_.features_ # Indices de las cols originales que generaron indicador
            indicator_col_names = [f"missing_{X_train.columns[i]}" for i in indicator_indices]
            feature_names_imputed.extend(indicator_col_names)
        if len(feature_names_imputed) != X_train_imputed_np.shape[1]: # Comprobación de seguridad
            print(f"Discrepancia en nombres de columnas: {len(feature_names_imputed)} vs {X_train_imputed_np.shape[1]}. Feature importance podría no tener nombres correctos.")
            feature_names_imputed = None # Forzar a usar array si los nombres no cuadran


    # Convertir de nuevo a DataFrames de Pandas
    if feature_names_imputed is not None:
        X_train_imputed = pd.DataFrame(X_train_imputed_np, columns=feature_names_imputed, index=X_train.index)
        X_test_imputed = pd.DataFrame(X_test_imputed_np, columns=feature_names_imputed, index=X_test.index)
    else: # Si no pudimos obtener los nombres, trabajamos con arrays de NumPy (menos ideal para feature importance explícita)
        X_train_imputed = X_train_imputed_np
        X_test_imputed = X_test_imputed_np
        print("Advertencia: X_train_imputed y X_test_imputed son arrays NumPy. Nombres de características no disponibles para la tabla de importancia.")


    print("\nManejo de NaNs después de la imputación (con indicadores):")
    # Si son DataFrames, podemos hacer isnull().sum().sum(). Si son arrays, no directamente.
    if isinstance(X_train_imputed, pd.DataFrame):
        print(f"NaNs en X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
        print(f"NaNs en X_test_imputed: {X_test_imputed.isnull().sum().sum()}")
    else: # Si son arrays NumPy, SimpleImputer debería haber manejado todos los NaNs
        print(f"NaNs en X_train_imputed (array NumPy): {np.isnan(X_train_imputed).sum()}")
        print(f"NaNs en X_test_imputed (array NumPy): {np.isnan(X_test_imputed).sum()}")

    print(f"Forma de X_train_imputed: {X_train_imputed.shape} (puede tener más columnas debido a los indicadores)")
    print(f"Forma de X_test_imputed: {X_test_imputed.shape}")
else:
    print("Error: X_train o X_test no están definidos. Ejecuta los pasos anteriores.")

## 3. Entrenar Random Forest Detector

In [ ]:
if 'X_train_imputed' in locals() and 'y_train' in locals():
    # Inicializar el clasificador Random Forest
    # class_weight='balanced' puede ayudar si hay desbalance entre clases 'normal' y 'anomalia'
    # n_estimators: número de árboles en el bosque.
    # random_state: para reproducibilidad.
    rf_detector = RandomForestClassifier(n_estimators=100,
                                            random_state=42,
                                            class_weight='balanced', # Útil si las clases están desbalanceadas
                                            n_jobs=-1) # Usar todos los procesadores disponibles

    print("\nEntrenando el modelo Random Forest Detector...")
    rf_detector.fit(X_train_imputed, y_train)
    print("Entrenamiento completado.")

else:
    print("Error: X_train_imputed o y_train no están definidos.")

## 4. Evaluar en test: métricas globales

In [ ]:
if 'rf_detector' in locals() and 'X_test_imputed' in locals() and 'y_test' in locals():
    print("\nRealizando predicciones en el conjunto de prueba...")
    y_pred_detector = rf_detector.predict(X_test_imputed)
    y_pred_proba_detector = rf_detector.predict_proba(X_test_imputed)[:, 1] # Probabilidades para la clase 'anomalia'

    print("\n--- Evaluación del Modelo Detector ---")
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred_detector)
    print(f"Accuracy: {accuracy:.4f}")

    # Matriz de Confusión
    print("\nMatriz de Confusión:")
    # [[VN, FP],
    #  [FN, VP]]
    # VN (True Negative): Normales (no inyectados) correctamente identificados como normales por Autoencoder.
    # FP (False Positive): Normales (no inyectados) incorrectamente identificados como anomalías por Autoencoder.
    # FN (False Negative): Anomalías inyectadas incorrectamente identificadas como normales por Autoencoder.
    # VP (True Positive): Anomalías inyectadas correctamente identificadas como anomalías por Autoencoder.
    labels_orden  = ['normal', 'anomalia']
    display_names = ['Normal', 'Anomalía']
    cm = confusion_matrix(y_test, y_pred_detector, labels=labels_orden)
    print(cm)
    # Visualizar la matriz de confusión
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=display_names, yticklabels=display_names)
    plt.xlabel('Predicción')
    plt.ylabel('Valor Real')
    plt.title('Matriz de Confusión - Detector de Anomalías')
    plt.show()

    # Desempaquetar la matriz de confusión para un análisis más detallado
    tn, fp, fn, tp = cm.ravel()
    print(f"\nAnálisis Adicional del Detector:")
    print(f"  - Verdaderos Positivos (VP - Anomalías detectadas): {tp}")
    print(f"  - Falsos Positivos (FP - Normales marcados como anomalías): {fp}")
    print(f"  - Falsos Negativos (FN - Anomalías NO detectadas): {fn}")
    print(f"  - Verdaderos Negativos (TN - Normales correctamente identificados): {tn}")

    # Reporte de Clasificación (Precisión, Recall, F1-score)
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred_detector, labels=labels_orden, target_names=display_names))

    # ROC AUC Score
    # (Requiere las probabilidades de la clase positiva)
    idx_anomalia = list(rf_detector.classes_).index('anomalia')
    y_pred_proba_anomalia = rf_detector.predict_proba(X_test_imputed)[:, idx_anomalia]
    y_bin = (y_test == 'anomalia').astype(int)
    roc_auc = roc_auc_score(y_bin, y_pred_proba_anomalia)
    pr_auc  = average_precision_score(y_bin, y_pred_proba_anomalia)
    print(f"ROC AUC Score:                            {roc_auc:.4f}")
    print(f"Precision-Recall AUC (Average Precision): {pr_auc:.4f}")

else:
    print("Error: El modelo 'rf_detector' no está entrenado o los datos de prueba no están disponibles.")

## 5. Importancia de features
¿Qué variables son más relevantes para la detección?

In [ ]:
if ('rf_detector' in locals() and
    hasattr(rf_detector, 'feature_importances_') and
    'X_train_imputed' in locals()): # Verificar X_train_imputed

    if not isinstance(X_train_imputed, pd.DataFrame):
        print("Error: X_train_imputed no es un DataFrame de Pandas. No se pueden obtener nombres de características.")
        print("Esto puede suceder si la construcción de feature_names_imputed falló en el paso de imputación.")
    else:
        importances = rf_detector.feature_importances_
        feature_names = X_train_imputed.columns
        
        # Verificar la longitud para evitar el error
        if len(feature_names) == len(importances):
            # Crear un DataFrame para visualizar mejor
            feature_importance_df = pd.DataFrame({'Característica': feature_names, 'Importancia': importances})
            feature_importance_df = feature_importance_df.sort_values(by='Importancia', ascending=False)
            
            print("\n--- Importancia de las Características (Detector de Anomalías) ---")
            print(feature_importance_df.to_string())
            
            # Graficar la importancia de las características
            plt.figure(figsize=(12, max(6, len(feature_names) * 0.4))) 
            plt.barh(feature_importance_df['Característica'], feature_importance_df['Importancia'], color='mediumpurple')
            plt.xlabel("Importancia Relativa")
            plt.ylabel("Característica")
            plt.title("Importancia de las Características - Detector de Anomalías (Random Forest)")
            plt.gca().invert_yaxis() 
            plt.tight_layout() 
            plt.show()
        else:
            print(f"Error de coincidencia de longitud: Nombres de características ({len(feature_names)}) vs. Importancias ({len(importances)}).")
            print("Asegúrate de que X_train_imputed tenga los nombres de columna correctos después de la imputación.")

else:
    print("Error: No se pudo calcular la importancia de las características.")
    print("Asegúrate de que 'rf_detector' esté entrenado y 'X_train_imputed' (como DataFrame con columnas) esté disponible.")

## 6. Desglose de detección por tipo de anomalía
¿Cuántas anomalías de cada tipo detecta correctamente el modelo?

In [ ]:
if ('df_trabajo' in locals() and
    'X_test' in locals() and
    'y_test' in locals() and
    'y_pred_detector' in locals()):

    true_anomaly_types_in_test_set = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']

    df_eval_detector = pd.DataFrame({
        'indice_original': X_test.index,
        'y_real_binaria': y_test,          # 'normal' o 'anomalia'
        'y_pred_detector': y_pred_detector, # 'normal' o 'anomalia' predicho
        'tipo_anomalia_real': true_anomaly_types_in_test_set
    })

    print("\n--- Desglose de Detección del Modelo 1 por Tipo Real de Anomalía (en Conjunto de Prueba) ---")

    df_anomalias_reales_en_test = df_eval_detector[df_eval_detector['y_real_binaria'] == 'anomalia']

    if df_anomalias_reales_en_test.empty:
        print("No hay anomalías reales en el conjunto de prueba para analizar.")
    else:
        detection_summary = df_anomalias_reales_en_test.groupby('tipo_anomalia_real').agg(
            total_instancias_reales_en_test = pd.NamedAgg(column='y_pred_detector', aggfunc='count'),
            correctamente_detectadas_como_anomalia = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'anomalia').sum()),
            no_detectadas_por_el_detector_FN = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'normal').sum())
        ).reset_index()

        detection_summary['tasa_deteccion_%'] = (
            detection_summary['correctamente_detectadas_como_anomalia'] /
            detection_summary['total_instancias_reales_en_test'] * 100
        )
        detection_summary = detection_summary.sort_values(by='tasa_deteccion_%', ascending=False)

        print("\nResumen de detección por tipo de anomalía real:")
        print(detection_summary.to_string())

        if not detection_summary.empty:
            plt.figure(figsize=(12, max(6, len(detection_summary) * 0.5)))
            detection_summary.set_index('tipo_anomalia_real')[['correctamente_detectadas_como_anomalia', 'no_detectadas_por_el_detector_FN']].plot(
                kind='barh', stacked=True,
                figsize=(12, max(6, len(detection_summary) * 0.6)),
                colormap='viridis'
            )
            plt.title('Rendimiento del Detector por Tipo de Anomalía Real (en Conjunto de Prueba)')
            plt.xlabel('Número de Instancias')
            plt.ylabel('Tipo de Anomalía Real')
            plt.legend(title='Resultado Detección', loc='lower right')
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(10, max(5, len(detection_summary) * 0.5)))
            sns.barplot(x='tasa_deteccion_%', y='tipo_anomalia_real', data=detection_summary,
                        palette='magma', hue='tipo_anomalia_real', dodge=False, legend=False)
            plt.title('Tasa de Detección del Modelo 1 por Tipo de Anomalía Real (%)')
            plt.xlabel('Tasa de Detección (%)')
            plt.ylabel('Tipo de Anomalía Real')
            plt.xlim(0, 105)
            for index, row in detection_summary.iterrows():
                plt.text(row['tasa_deteccion_%'] + 1, index,
                         f"{row['tasa_deteccion_%']:.1f}%", color='black', ha="left", va="center")
            plt.tight_layout()
            plt.show()

else:
    print("Error: No se pueden generar los datos de desglose.")
    print("Asegúrate de que 'df_trabajo', 'X_test', 'y_test', y 'y_pred_detector' estén disponibles.")


## 7. Guardar modelo y resultados

In [ ]:
os.makedirs(DATA_MODELS, exist_ok=True)
joblib.dump(rf_detector,     MODELO_1_PATH)
joblib.dump(imputer,         IMPUTER_M1_PATH)
joblib.dump(list(X_train_imputed.columns), FEATURES_M1_PATH)
print(f"Modelo 1 guardado: {MODELO_1_PATH}")

# Guardar predicciones sobre todo el dataset para el Modelo 2
X_full_imputed = pd.DataFrame(
    imputer.transform(X),
    columns=imputer.get_feature_names_out(X.columns)
)
df_trabajo['pred_deteccion'] = rf_detector.predict(X_full_imputed)
df_trabajo.to_parquet(PARQUET_04, index=False)
print(f"Predicciones guardadas: {PARQUET_04}")
